# 12 - Clasificacion CT a nivel de estudio

Este notebook anade una fase final para CT. La motivacion es que MosMedData etiqueta estudios completos, no slices individuales. Por eso, entrenar un modelo 2D slice a slice puede introducir ruido: un corte puede parecer normal, aunque el estudio completo tenga una etiqueta de severidad.

Aqui no se entrena otro backbone pesado. Se reutiliza el mejor modelo CT con slices informativos (`top20_tissue + ResNet50 + weighted CE`) como extractor de probabilidades por slice. Despues se construye una representacion por estudio y se entrena un meta-clasificador ligero a nivel de estudio.

## Que hace esta fase

1. Carga el modelo `ct_top20_tissue_resnet50_weighted_ce` ya entrenado.
2. Genera probabilidades por slice para train, validacion y test.
3. Agrupa los slices por `study_id`.
4. Resume cada estudio con estadisticos: media, maximo, percentiles, votos por clase, confianza y entropia.
5. Ajusta candidatos sencillos en train y selecciona el mejor en validacion.
6. Evalua una unica vez en test.

La idea no es afirmar que esto sustituye a un modelo 3D, sino crear una aproximacion intermedia y reproducible entre el modelo 2D y una futura estrategia volumetrica.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd
from IPython.display import Image, display, Markdown

KNOWN_PROJECT_ROOT = Path('/Users/yuncaichen/Master/TFM:FINAL/TFM')

def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents, KNOWN_PROJECT_ROOT]
    for candidate in candidates:
        if (candidate / 'src').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('No se encontro la raiz del proyecto TFM.')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / 'results' / '.matplotlib'))

PYTHON_BIN = PROJECT_ROOT / '.conda' / 'bin' / 'python'
if not PYTHON_BIN.exists():
    PYTHON_BIN = Path(sys.executable)

RESULTS_DIR = PROJECT_ROOT / 'results' / 'classification' / 'ct_study_level_meta'
FIGURES_DIR = RESULTS_DIR / 'figures'
PROJECT_ROOT

## Ejecutar el meta-clasificador por estudio

Esta celda ejecuta el script completo. Si ya existen predicciones por slice guardadas, las reutiliza. Si quieres forzar que se recalculen desde el modelo, deja `FORCE_INFERENCE = True`.

In [ ]:
RUN_STUDY_LEVEL_META = True
FORCE_INFERENCE = False

command = [
    str(PYTHON_BIN),
    str(PROJECT_ROOT / 'scripts' / 'run_ct_study_level_meta_classifier.py'),
    '--base-experiment', 'ct_top20_tissue_resnet50_weighted_ce',
    '--batch-size', '32',
    '--num-workers', '0',
]
if FORCE_INFERENCE:
    command.append('--force-inference')

if RUN_STUDY_LEVEL_META:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('Ejecucion desactivada. Cambia RUN_STUDY_LEVEL_META = True para lanzarlo.')
    print(' '.join(command))

## Resultados principales

La tabla compara reglas simples de agregacion con el mejor meta-clasificador seleccionado en validacion. Para CT, miramos especialmente F1-macro porque el dataset esta desbalanceado y no queremos que el resultado quede dominado por la clase mayoritaria.

In [ ]:
metrics_path = RESULTS_DIR / 'ct_study_level_meta_metrics.csv'
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
    display(
        metrics_df[metrics_df['split'].eq('test')]
        .sort_values(['f1_macro', 'auc_roc_macro'], ascending=False)
        [['method', 'n_studies', 'accuracy', 'f1_macro', 'f1_weighted', 'auc_roc_macro']]
        .round(4)
    )
else:
    print('Todavia no existe la tabla de metricas. Ejecuta la celda anterior.')

## Candidatos evaluados en validacion

Esta tabla no es el resultado final de test. Sirve para ver que candidato fue elegido antes de mirar test. Asi se mantiene una separacion metodologica correcta entre seleccion y evaluacion final.

In [ ]:
validation_path = RESULTS_DIR / 'ct_study_level_meta_validation_candidates.csv'
if validation_path.exists():
    validation_df = pd.read_csv(validation_path)
    display(
        validation_df.sort_values(['selected_for_test', 'f1_macro', 'auc_roc_macro'], ascending=False)
        [['method', 'selected_for_test', 'accuracy', 'f1_macro', 'f1_weighted', 'auc_roc_macro']]
        .round(4)
    )
else:
    print('Todavia no existe la tabla de validacion.')

## Figuras

La primera figura compara F1-macro y AUC macro en test. La segunda muestra la matriz de confusion del mejor metodo de test.

In [ ]:
for figure_name in [
    'ct_study_level_meta_test_metrics.png',
    'ct_study_level_meta_best_confusion_matrix.png',
]:
    figure_path = FIGURES_DIR / figure_name
    if figure_path.exists():
        print(figure_path.relative_to(PROJECT_ROOT))
        display(Image(filename=str(figure_path)))
    else:
        print(f'No existe todavia: {figure_path.relative_to(PROJECT_ROOT)}')

## Lectura para el TFM

Si el meta-clasificador mejora las reglas simples, se puede defender que resumir el volumen con informacion de varios slices aporta senal adicional. Si no mejora, tambien es una conclusion util: indicaria que la media de probabilidades ya captura casi toda la informacion que el modelo 2D extrae, y que una mejora real probablemente necesita contexto 3D o anotaciones mas precisas.

En cualquier caso, esta fase mejora el rigor del TFM porque evalua CT en la unidad correcta: el estudio completo.